# Assignment 1
# Simple Agentic AI Demo with Microsoft AutoGen

In [1]:
# ===== BASIC IMPORTS =====
from dataclasses import dataclass
import os

from autogen_core import (
    MessageContext,
    RoutedAgent,
    message_handler,
    AgentId,
    SingleThreadedAgentRuntime,
)

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
# ===== MESSAGE TYPE =====
@dataclass
class MyMessageType:
    content: str

In [4]:
# ===== MODEL CLIENT =====
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",    # or whichever model you prefer
    api_key=os.environ["OPENAI_API_KEY"],
)

In [5]:
# ===== FIRST AGENT DEFINITION =====
class MyFirstAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        self.name = name
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: MyMessageType, ctx: MessageContext) -> None:
        print(f"[{self.name}] received:", message.content)

        # Use LLM to craft a short reply via AgentChat API
        result = await self._delegate.run(
            task=(
                "You are Agent 1 in a two-agent demo. "
                "Reply briefly and clearly so that Agent 2 can read your answer.\n\n"
                f"User message: {message.content}"
            )
        )

        reply_text = result.messages[-1].content  # AgentChat TaskResult
        print(f"[{self.name}] replied:", reply_text)

        # Send the reply as a new MyMessageType to MySecondAgent
        await self.send_message(
            MyMessageType(content=reply_text),
            AgentId("agent2", "default"),
        )

# ===== SECOND AGENT DEFINITION =====
class MySecondAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        self.name = name
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: MyMessageType, ctx: MessageContext) -> None:
        print(f"[{self.name}] received:", message.content)

        # Use LLM to respond to Agent 1's message
        result = await self._delegate.run(
            task=(
                "You are Agent 2 in a two-agent demo. "
                "Reply briefly to the message you just received from Agent 1.\n\n"
                f"Message from Agent 1: {message.content}"
            )
        )

        reply_text = result.messages[-1].content
        print(f"[{self.name}] replied:", reply_text)


In [6]:
runtime = SingleThreadedAgentRuntime()

await MyFirstAgent.register(runtime, "agent1", lambda: MyFirstAgent("agent1"))
await MySecondAgent.register(runtime, "agent2", lambda: MySecondAgent("agent2"))

runtime.start()

await runtime.send_message(
    MyMessageType("Hello Agent 1, please start the conversation with Agent 2."),
    AgentId("agent1", "default"),
)

import asyncio
await asyncio.sleep(3)
await runtime.stop()



[agent1] received: Hello Agent 1, please start the conversation with Agent 2.
[agent1] replied: Hello Agent 2, how are you today?
[agent2] received: Hello Agent 2, how are you today?
[agent2] replied: Hello Agent 1! I'm doing well, thank you. How about you?
